In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window, DataFrame
import torch

torch.manual_seed(123)

src_path = Path.cwd().parent / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to sys.path: {src_path}")

load_dotenv()  # reads .env file from the current directory
spark = SparkSession.builder.getOrCreate()

# constants

In [ ]:
path_narratives = "data/faers/faers_narratives_small"

model_weights_settings = Path.cwd().parent / "data" / "model_weights" / "gpt2" / "124M" / "settings.pickle.gz"
model_weights_parameters = Path.cwd().parent / "data" / "model_weights" / "gpt2" / "124M" / "parameters.pickle.gz"

SEQUENCE_SEPARATOR = "<|endoftext|>"  # [EOS]

DEFAULT_VOCABULARY_SIZE = 50257
DEFAULT_EMBEDDINGS_DIMENSION = 256
DEFAULT_SEQUENCE_LENGTH = 4
DEFAULT_STRIDE = 1

# data

In [ ]:
def get_corpus_subset(subset_length: int = 10000) -> str:
    df_narratives = spark.read.load(path_narratives)
    result: str = SEQUENCE_SEPARATOR.join(df_narratives.limit(subset_length).select("text").toPandas()["text"].tolist())
    return result

In [ ]:
from tgedr_languagemodels.utils_llm import save_pickle_compressed, load_pickle_compressed

params = load_pickle_compressed(model_weights_parameters)
settings = load_pickle_compressed(model_weights_settings)

# model

In [ ]:
from tgedr_languagemodels.models import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"

NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024})
NEW_CONFIG.update({"qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

In [ ]:
from tgedr_languagemodels.model_weights import load_weights_into_gpt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_weights_into_gpt(gpt, params)
gpt.to(device)

In [ ]:
from tgedr_languagemodels.utils_llm import generate, text_to_token_ids, token_ids_to_text
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")


token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5,
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

# fine-tuning for classification